# Xenium sc-SVC Fibroblast Reconstruction Impact

A single-route, reproducible post-reconstruction analysis.

## 1. Question and route semantics

Raw is the original spatial expression representation. The spatial carrier is expression-identical to Raw on the shared axis, so it is audited but not treated as an impact edge; the only edge is Raw Leiden to final SVC.

**Evidence boundary.** This notebook quantifies paired representation and spatial-pattern changes; it does not establish biological mechanism, truth, or clinical relevance.

In [ ]:
import hashlib
import os
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from revise.analysis.reconstruction_impact import (compute_spatial_impact, file_sha256, load_reconstruction_impact_config, run_partition_analysis, write_analysis_artifacts, write_partition_artifacts, write_spatial_artifacts)

ROOT = Path.cwd().resolve()
if not (ROOT / "configs" / "analysis").is_dir():
    raise RuntimeError("Run from the REVISE repository root.")
sns.set_theme(style="white", context="notebook")

def coordinates(adata):
    return pd.DataFrame(np.asarray(adata.obsm["spatial"], dtype=float)[:, :2], index=adata.obs_names, columns=["x", "y"])

def save(fig, name):
    fig.savefig(OUTPUT_DIR / "figures" / f"{name}.png", dpi=180, bbox_inches="tight")
    plt.show(); plt.close(fig)

def window_plot(ax, table, value, title, cmap="viridis", centered=False):
    values = table[value]
    limits = {} if not centered else {"vmin": -max(float(np.nanmax(abs(values))), 1e-9), "vmax": max(float(np.nanmax(abs(values))), 1e-9)}
    image = ax.scatter(table.window_x, table.window_y, c=values, cmap=cmap, s=10, linewidths=0, **limits)
    ax.set(title=title, xlabel="x (um)", ylabel="y (um)", aspect="equal"); ax.invert_yaxis(); plt.colorbar(image, ax=ax, shrink=.7)


## 2. Input and spatial-scale audit

Observation IDs are audited strictly. Coordinate units are converted to microns before non-overlapping windows: 8 um cell-equivalent, 5× = 40 um primary window.

In [ ]:
CONFIG = load_reconstruction_impact_config(ROOT / "configs" / "analysis" / 'reconstruction_impact_xenium_p2crc_fibroblast.yaml')
OUTPUT_DIR = Path(os.environ.get("REVISE_ANALYSIS_OUTPUT_ROOT", CONFIG["output"]["dir"]))
if not OUTPUT_DIR.is_absolute(): OUTPUT_DIR = ROOT / OUTPUT_DIR
(OUTPUT_DIR / "figures").mkdir(parents=True, exist_ok=True)
comparison_config = CONFIG["partition_change"]["comparisons"][0]
RAW_PATH = ROOT / comparison_config["raw_h5ad"]
RECON_PATH = ROOT / comparison_config["reconstructed_spatial_h5ad"]
raw_context = ad.read_h5ad(RAW_PATH, backed="r")
reconstructed_context = ad.read_h5ad(RECON_PATH, backed="r")
reconstructed_ids = reconstructed_context.obs_names.copy()
if not set(reconstructed_ids) <= set(raw_context.obs_names):
    raise ValueError("Reconstructed IDs must occur in Raw.")
level1_col, spatial = comparison_config["level1_column"], CONFIG["spatial_region"]
raw_full_coords = coordinates(raw_context)
raw_full_level1 = raw_context.obs[level1_col].astype(str).copy()
raw_full_n_units, raw_full_n_genes = raw_context.n_obs, raw_context.n_vars
sample_n = CONFIG["partition_change"].get("sample_n_units")
if not sample_n:
    raw_paired = raw_context[reconstructed_ids].to_memory()
    reconstructed = reconstructed_context.to_memory()
    raw_context.file.close(); reconstructed_context.file.close()
print(f"Raw context={raw_full_n_units:,}; strict paired cohort={len(reconstructed_ids):,}; scale={spatial['microns_per_coordinate']} um/unit")


## 3. Expression-state partition impact

Hungarian matching aligns Leiden labels. `1-matched accuracy` is ST-unit change and `1-matched macro-F1` is balanced cluster change; ARI, AMI, VI, and NMI are diagnostics.

In [ ]:
seed = int(CONFIG["partition_change"]["random_state"])
if sample_n:
    ids = pd.Index(np.random.default_rng(seed).choice(reconstructed_ids.to_numpy(), size=int(sample_n), replace=False)).sort_values()
    raw_partition, recon_partition = raw_context[ids].to_memory(), reconstructed_context[ids].to_memory()
    sampling_mode = f"deterministic_same_id_sample_{sample_n}"
else:
    raw_partition, recon_partition, sampling_mode = raw_paired, reconstructed, "full_paired_cohort"
partition = run_partition_analysis(raw_partition, recon_partition, level1_col=level1_col, final_cluster_key=comparison_config.get("reconstructed_cluster_key"), route_kind=CONFIG["route_kind"], resolution_mode=CONFIG["partition_change"]["mode"], resolution_candidates=CONFIG["partition_change"]["level1_resolution_candidates"], within_level1_resolution=CONFIG["partition_change"]["within_level1_resolution"], random_state=seed, n_top_genes=CONFIG["partition_change"]["n_top_genes"])
edge_name, comparison = next(iter(partition.comparisons.items()))
display(comparison.summary); display(partition.sweep)
if partition.representation_audit: display(pd.DataFrame([partition.representation_audit]))
if sample_n:
    raw_paired = raw_context[reconstructed_ids, partition.feature_names].to_memory()
    reconstructed = reconstructed_context[reconstructed_ids, partition.feature_names].to_memory()
    raw_context.file.close(); reconstructed_context.file.close()
    spatial_partition = run_partition_analysis(raw_paired, reconstructed, level1_col=level1_col, route_kind=CONFIG["route_kind"], resolution_mode="fixed_within_level1", within_level1_resolution=partition.resolution, random_state=seed, feature_names=partition.feature_names)
else:
    spatial_partition = partition
spatial_comparison = next(iter(spatial_partition.comparisons.values()))


## 4. Level1 anatomy candidates

Tumor and Normal (`Intestinal Epithelial`) are fixed context candidates. Their union assigns Tumor, Normal, Interface, and Other; anatomy is not a reconstruction-change endpoint.

## 5. Spatial partition change

Matched `unit_changed` is aggregated into the same physical windows.

## 6. Local diversity and Region

Internal metrics compute Raw Neff, reconstructed Neff, and Delta Neff. The primary Region is valid windows with reconstructed Neff at least 2; Delta Neff is diagnostic only.

In [ ]:
assignments = spatial_comparison.assignments.set_index("unit_id")
impact = compute_spatial_impact(full_coordinates=raw_full_coords, full_level1_labels=raw_full_level1, paired_coordinates=raw_full_coords.loc[raw_paired.obs_names], raw_labels=assignments.raw_cluster, reconstructed_labels=assignments.recon_cluster, unit_changed=assignments.unit_changed, microns_per_coordinate=float(spatial["microns_per_coordinate"]), cell_equivalent_um=float(spatial["cell_equivalent_um"]), main_window_multiplier=int(spatial["main_window_multiplier"]), window_multipliers=list(spatial["window_multipliers"]), neff_threshold=float(spatial["neff_threshold"]), neff_thresholds=list(spatial["neff_thresholds"]), tumor_label=spatial["anatomy_region"]["tumor_label"], normal_source_label=spatial["anatomy_region"]["normal_source_label"])
input_audit = pd.DataFrame([{"input":"raw_context","path":str(RAW_PATH),"sha256":file_sha256(RAW_PATH),"n_units":raw_full_n_units,"n_genes":raw_full_n_genes},{"input":"reconstructed_spatial","path":str(RECON_PATH),"sha256":file_sha256(RECON_PATH),"n_units":reconstructed.n_obs,"n_genes":reconstructed.n_vars}])
manifest = {"sample_id":CONFIG["sample"]["id"],"route_kind":CONFIG["route_kind"],"comparison_edge":edge_name,"paired_units":int(raw_paired.n_obs),"excluded_raw_units":int(raw_full_n_units-raw_paired.n_obs),"pairing_status":"strict_same_id","sampling_mode":sampling_mode,"random_state":seed,"resolution":partition.resolution,"resolution_source":partition.resolution_source,"resolution_sweep":partition.sweep.to_dict(orient="records"),"representation_audit":partition.representation_audit}
write_analysis_artifacts(OUTPUT_DIR, config=CONFIG, manifest=manifest, input_audit=input_audit)
write_partition_artifacts(OUTPUT_DIR, partition); write_spatial_artifacts(OUTPUT_DIR, impact)
display(pd.DataFrame([impact.scale_audit | impact.support_selection])); display(impact.anatomy_summary)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
normalized = comparison.contingency.div(comparison.contingency.sum(axis=1), axis=0).fillna(0)
sns.heatmap(normalized, ax=axes[0], cmap="mako", vmin=0, vmax=1); axes[0].set(title="Matched normalized contingency", xlabel="Reconstructed / Final", ylabel="Raw Leiden")
summary = comparison.summary.iloc[0]
axes[1].bar(["ST-unit change", "Balanced change"], [summary.st_unit_change_fraction, summary.balanced_cluster_change], color=["#d95f02", "#7570b3"]); axes[1].set(ylim=(0,1), ylabel="fraction", title="Partition-change complements")
save(fig, "01_partition_change")

anatomy = impact.anatomy_windows
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, column, label, cmap in zip(axes, ["tumor_candidate","normal_candidate","union_candidate","level1_region"], ["Tumor candidate","Normal candidate","Tumor/Normal union","Level1 anatomy"], ["Reds","Blues","viridis","tab10"]):
    value = pd.Categorical(anatomy[column].astype(str)).codes if column == "level1_region" else anatomy[column]
    ax.scatter(anatomy.window_x, anatomy.window_y, c=value, cmap=cmap, s=8, linewidths=0); ax.set(title=label, xlabel="x (um)", ylabel="y (um)", aspect="equal"); ax.invert_yaxis()
save(fig, "02_level1_anatomy_candidates")

frame = impact.unit_assignments.sample(min(len(impact.unit_assignments), 50000), random_state=seed)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, column, label in zip(axes[:2], ["raw_cluster","reconstructed_cluster"], ["Raw Leiden clusters","Reconstructed / Final clusters"]):
    ax.scatter(frame.x, frame.y, c=pd.Categorical(frame[column]).codes, cmap="tab20", s=1, linewidths=0); ax.set(title=label, xlabel="x (um)", ylabel="y (um)", aspect="equal"); ax.invert_yaxis()
window_plot(axes[2], impact.window_metrics, "unit_change_fraction", "Window unit-change fraction", cmap="magma")
save(fig, "03_spatial_partition_change")

fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
window_plot(axes[0], impact.window_metrics, "neff_raw", "Raw Neff")
window_plot(axes[1], impact.window_metrics, "neff_recon", "Reconstructed Neff")
window_plot(axes[2], impact.window_metrics, "delta_neff", "Centered Delta Neff", cmap="coolwarm", centered=True)
window_plot(axes[3], impact.window_metrics.assign(region_mask=impact.window_metrics.in_region.astype(int)), "region_mask", "High-diversity Region", cmap="Greens")
save(fig, "04_local_diversity_and_region")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
support = impact.support_sensitivity
if support.empty: axes[0].text(.5,.5,"Insufficient parent support",ha="center",va="center")
else:
    axes[0].plot(support.min_parent_units, support.valid_window_fraction, label="valid windows"); axes[0].plot(support.min_parent_units, support.retained_unit_fraction, label="retained units"); axes[0].axvline(impact.support_selection["min_parent_units"], color="black", linestyle="--"); axes[0].legend()
axes[0].set(xlabel="minimum parent units", ylabel="fraction", title="Support-selection knee")
axes[1].plot(impact.scale_sensitivity.window_multiplier, impact.scale_sensitivity.region_unit_fraction, marker="o"); axes[1].set(xlabel="cell-equivalent multiplier", ylabel="Region unit fraction", title="Scale sensitivity")
axes[2].plot(impact.threshold_sensitivity.threshold, impact.threshold_sensitivity.region_area_fraction, marker="o"); axes[2].set(xlabel="Neff threshold", ylabel="Region area fraction", title="Threshold sensitivity")
save(fig, "05_sensitivity")


## 7. Sensitivity and take-home results

Scale sensitivity holds the main-scale support selection fixed; threshold sensitivity varies only the Neff cutoff.

In [ ]:
primary = comparison.summary.iloc[0]
take_home = pd.DataFrame([{"comparison_edge":edge_name,"ST-unit change":primary.st_unit_change_fraction,"balanced cluster change":primary.balanced_cluster_change,"ARI":primary.ARI,"selected parent support":impact.support_selection["min_parent_units"]}])
display(take_home)
print("These are paired representation and spatial-pattern results, not biological mechanism or truth validation.")
